# Lab 11 — Visualisation Studio
**Charts That Communicate Track** · Beginner · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Choose appropriate charts for distribution and composition questions
2. Redact PII (emails) before any plot leaves your machine
3. Produce four annotated publication-quality figures
4. Read tip% and ticket-status insights from the charts

## Datasets (this folder)
- `tips.csv` — auto-download from `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv`
- `tickets.csv` — auto-download from `https://raw.githubusercontent.com/vihar/datasets/master/tickets.csv`

## How to run on Google Colab
1. Click **Start Lab** — the hosted notebook opens directly in Colab under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0 (bootstrap)** first — it pulls `dataset.zip` from the lab manifest into `/content/ml_lab` (falls back to public raw URLs, then local files).
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 fetches `dataset.zip` from the manifest → `Runtime → Run all`.


### Setup (VLABS bootstrap)

Run the next cell (Cell 0) once. Fetch order: hosted `manifest.json` → `dataset.zip` extracted to `/content/ml_lab/<lab_id>` → per-file public raw URLs → local files next to this notebook. No-op when files already exist.


In [ ]:
# Cell 0 — VLABS bootstrap: run first. Works on Colab (direct-open URL) and locally.
import io, json, os, urllib.request, zipfile

LAB_ID = "lab-11-visualisation-studio"
# Hosted manifest (Admin: replace ORG/REPO once per deployment).
MANIFEST_URL = f"https://raw.githubusercontent.com/ORG/REPO/main/{LAB_ID}/manifest.json"
# Alternative: backend proxy to S3 — uncomment to use instead:
# MANIFEST_URL = f"https://api.vlabs.test/colab/{LAB_ID}/manifest"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = f"/content/ml_lab/{LAB_ID}" if ON_COLAB else "."

def _fetch(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()

def _ensure_file(filename, url=None):
    """Local-first single-file fetch (also used by lesson load cells)."""
    for base in (DATA_DIR, "."):
        p = os.path.join(base, filename)
        if os.path.exists(p):
            print(f"found {p}")
            return p
    if not url:
        raise FileNotFoundError(
            f"{filename} missing: open via Start Lab (bundle) or add it next to the notebook")
    os.makedirs(DATA_DIR, exist_ok=True)
    dest = os.path.join(DATA_DIR, filename)
    print(f"downloading {filename} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"saved {dest}")
    return dest

ensure = _ensure_file  # compat alias for lesson load cells

def vlabs_bootstrap():
    # 1) Hosted manifest -> dataset.zip -> DATA_DIR (direct-open path)
    try:
        m = json.loads(_fetch(MANIFEST_URL).decode("utf-8"))
        dz = m.get("dataset_zip")
        if dz:
            print(f"manifest ok: {MANIFEST_URL}")
            os.makedirs(DATA_DIR, exist_ok=True)
            zpath = os.path.join(DATA_DIR, "dataset.zip")
            urllib.request.urlretrieve(dz, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(DATA_DIR)
            print(f"extracted dataset.zip -> {DATA_DIR}")
    except Exception as e:
        print(f"manifest skip ({e}); using file fallbacks")
    # 2) Per-file fallbacks (public raw URLs; local files are a no-op hit)
    _ensure_file("tickets.csv", "https://raw.githubusercontent.com/vihar/datasets/master/tickets.csv")
    _ensure_file("tips.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
    # 3) Work from the data dir on Colab so relative paths resolve
    if ON_COLAB and DATA_DIR != ".":
        os.chdir(DATA_DIR)
        print(f"cwd -> {DATA_DIR}")

vlabs_bootstrap()


## Charts That Communicate Track: Tips + Tickets (4 Publication-Quality Plots)

> **Scenario:** Two briefs, two datasets: (1) `tips.csv` — *who tips more?* (2) `tickets.csv` — *SLA status mix*; the tickets file contains **emails (PII)** so redact before any plot leaves your machine. Produce four charts with annotated insights.
>
> **You will learn:** figure anatomy, chart choice, PII redaction, subplots, annotation.
> **Time:** ~45 minutes. **Level:** Beginner. **Needs:** pandas + matplotlib. **Env:** 🟢 Colab only.

### Chart-choice mental map

| Question | Chart | Do not use |
|---|---|---|
| Distribution of a measure | histogram / box | pie for many bins |
| Share of categories | stacked/bar (few cats) | 3-D pie |
| Relationship bill vs tip | scatter | bar |
| Trend over time | line | scatter if ordered |

---

### 1. Load both datasets (local first, Colab fallback)

In [ ]:
import os, re
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def ensure(name, url):
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(url, name)
    return name

ensure("tips.csv",
       "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
ensure("tickets.csv",
       "https://raw.githubusercontent.com/vihar/datasets/master/tickets.csv")

tips = pd.read_csv("tips.csv")
tickets = pd.read_csv("tickets.csv")
print(tips.shape, tickets.shape)  # (244, 7) (107, 9)
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
print(tips["tip_pct"].mean().round(4))  # 0.1608 overall
print(tickets["status"].value_counts())
# open 42, in-progress 34, closed 31


---

### 2. Redact PII before plotting tickets

In [ ]:
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")

def redact_email(s):
    return EMAIL_RE.sub("[EMAIL]", str(s))

# How much PII is in here?
raw_hits = sum(len(EMAIL_RE.findall(str(v))) for v in tickets.astype(str).values.ravel())
print("email-like hits before redaction:", raw_hits)  # 184+

tickets_clean = tickets.copy()
for col in tickets_clean.columns:
    tickets_clean[col] = tickets_clean[col].apply(
        lambda v: redact_email(v) if isinstance(v, str) and "@" in str(v) else v
    )

still = sum(len(EMAIL_RE.findall(str(v))) for v in tickets_clean.astype(str).values.ravel())
print("hits after redaction:", still)  # 0
assert still == 0


> Redaction is **pre-plot hygiene**. Don’t put raw emails in chart titles, legends, or exported PNGs.

---

### 3. Chart 1 — Tip % distribution by day (histogram overlay)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for day, sub in tips.groupby("day"):
    ax.hist(sub["tip_pct"], bins=15, alpha=0.45, label=f"{day} (n={len(sub)})")
ax.set_xlabel("Tip % of bill")
ax.set_ylabel("Count")
ax.set_title("Tip percentage distribution by day")
ax.legend()
ax.annotate("Overall mean ≈ 16.1%",
            xy=(0.1608, tips.groupby("day").size().max() * 0.45),
            fontsize=9)
fig.tight_layout(); fig.savefig("chart1_tip_pct_by_day.png", dpi=120)
print("saved chart1_tip_pct_by_day.png")


Mean tip % by day (for the caption): **Fri 17.0% · Thur 16.1% · Sun 16.7% · Sat 15.3%**.

---

### 4. Chart 2 — Mean tip % by day + time (grouped bar)

In [ ]:
pivot = tips.pivot_table(index="day", columns="time",
                         values="tip_pct", aggfunc="mean").reindex(["Thur", "Fri", "Sat", "Sun"])
print(pivot.round(4))

ax = pivot.plot(kind="bar", figsize=(7, 4), rot=0)
ax.set_ylabel("Mean tip %")
ax.set_title("Mean tip % by day and sitting")
ax.set_ylim(0, 0.25)
ax.legend(title="Sitting")
fig = ax.get_figure(); fig.tight_layout()
fig.savefig("chart2_tip_pct_day_time.png", dpi=120)
print("saved chart2_tip_pct_day_time.png")


Insight to annotate: lunch tips slightly higher on average (Lunch 16.4% vs Dinner 16.0% overall).

---

### 5. Chart 3 — Tickets: stacked status × priority (redacted data only)

In [ ]:
ct = pd.crosstab(tickets_clean["priority"], tickets_clean["status"])
# order columns for readability
for c in ["open", "in-progress", "closed"]:
    if c not in ct.columns:
        ct[c] = 0
ct = ct[["open", "in-progress", "closed"]]
print(ct)

ax = ct.plot(kind="bar", stacked=True, figsize=(7, 4), rot=0,
             color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_ylabel("Tickets")
ax.set_title("Ticket status mix by priority (PII redacted)")
ax.annotate("Open work concentrated in low/med priority",
            xy=(0.02, 0.92), xycoords="axes fraction", fontsize=9)
fig = ax.get_figure(); fig.tight_layout()
fig.savefig("chart3_tickets_status.png", dpi=120)
print("saved chart3_tickets_status.png")


Expected status totals: open 42 · in-progress 34 · closed 31 (n = 107).

---

### 6. Chart 4 — Total bill vs tip scatter (annotated insight)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = {"Male": "#4C72B0", "Female": "#C44E52"}
for sex, sub in tips.groupby("sex"):
    ax.scatter(sub["total_bill"], sub["tip"], s=28, alpha=0.7,
               label=f"{sex} (mean tip% {sub['tip_pct'].mean():.1%})",
               color=colors.get(sex, None))
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip ($)")
ax.set_title("Tip vs total bill")
ax.legend()
# annotate the largest bill
i = tips["total_bill"].idxmax()
ax.annotate(f"max bill ${tips.loc[i, 'total_bill']:.2f}",
            xy=(tips.loc[i, "total_bill"], tips.loc[i, "tip"]),
            xytext=(tips.loc[i, "total_bill"] - 12, tips.loc[i, "tip"] + 1.2),
            arrowprops=dict(arrowstyle="->", lw=0.8))
fig.tight_layout(); fig.savefig("chart4_tip_vs_bill.png", dpi=120)
print("saved chart4_tip_vs_bill.png")


Female mean tip % ≈ 16.7% · Male ≈ 15.8% (small gap, don’t over-claim).

---

## Exercises (do these!)

### Exercise 1 — Tip % distribution by day
Compute mean `tip_pct` by `day` (4 d.p.). Which day is highest? Lowest?
*Expected: highest Fri 0.1699 · lowest Sat 0.1532 (overall 0.1608).*

<details>
<summary>Hint</summary>

`tips["tip_pct"] = tips["tip"]/tips["total_bill"]` then `groupby("day")["tip_pct"].mean()`.
</details>

### Exercise 2 — Stacked status bar for tickets
Build `pd.crosstab(priority, status)` on **redacted** tickets and plot stacked bars. Print the table.
*Expected columns sum: open 42, in-progress 34, closed 31; priority rows low 41, medium 34, high 32.*

<details>
<summary>Hint</summary>

Redact first (Section 2), then `crosstab`, then `.plot(kind="bar", stacked=True)`.
</details>

### Exercise 3 — Annotate insight on each chart
For each of the 4 saved PNGs, add at least one `ax.annotate(...)` or title subtitle stating the key insight (numbers included). List the four insight strings you used.
*Expected: any four insights citing computed stats — e.g. “Fri mean tip 17.0%”, “42 open tickets”, “overall tip 16.1%”, “max bill $50.81”.*

<details>
<summary>Hint</summary>

`ax.annotate("text", xy=(x,y), fontsize=9)` or include numbers in `set_title`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
by_day = tips.groupby("day")["tip_pct"].mean().sort_values(ascending=False)
print(by_day.round(4))
# Fri    0.1699  (highest)
# Sun    0.1669
# Thur   0.1613
# Sat    0.1532  (lowest)

# --- Solution 2 ---
# tickets_clean already redacted in Section 2
ct = pd.crosstab(tickets_clean["priority"], tickets_clean["status"])
print(ct)
print("col sums:", ct.sum().to_dict())
# expect open=42, in-progress=34, closed=31

# --- Solution 3 --- (example annotations used above)
insights = [
    "Overall mean tip ≈ 16.1% across 244 bills",
    "Lunch mean tip% 16.4% slightly above Dinner 16.0%",
    "42 open / 34 in-progress / 31 closed (PII redacted)",
    f"Max bill ${tips['total_bill'].max():.2f} still ~15–20% tip band",
]
for s in insights:
    print("-", s)


### What to learn next
- seaborn `boxplot` / `violinplot` for group distributions.
- Consistent style: `plt.rcParams`, colourblind-safe palettes.
- Export for slides: `dpi=300`, tight bbox, no chartjunk.
- Cheat sheet: redact PII → pick chart → compute the number → plot → annotate the insight.

*Files in this folder: `tips.csv`, `tickets.csv` · outputs `chart1_tip_pct_by_day.png`, `chart2_tip_pct_day_time.png`, `chart3_tickets_status.png`, `chart4_tip_vs_bill.png`.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
